In [8]:
!pip install -q langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-groq faiss-cpu sentence-transformers

In [9]:
import os
from google.colab import userdata

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_groq import ChatGroq

In [10]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [11]:
tricky_document = """
Section 1: The company Jade Global is launching a massive new
internal initiative called Project Phoenix. This project will
restructure the entire cloud infrastructure.
Section 2: Employees must adhere to the standard office hours of
9:00 AM to 5:00 PM. Remote work is permitted on Tuesdays and
Thursdays, provided that the employee has secured prior approval
from their direct manager.
Section 3: The cafeteria will now offer extended hours, opening
at 7:30 AM for breakfast. Please ensure you clear your tables
after eating.
Section 4: All IT support tickets must be filed through the
internal Jira portal. Direct emails to the IT staff will be
ignored starting next month.
Section 5: The annual holiday party is scheduled for December
15th. Dress code is semi-formal. Plus-ones are allowed if
registered by November 30th.
Section 6: Parking in the executive lot is strictly prohibited
for unauthorized vehicles. Violators will be towed at the owner's
expense.
Section 7: Health insurance open enrollment begins in October.
Please review the new dental and vision plans, as the providers
have changed this year.
Section 8: All employees must complete the mandatory
cybersecurity training module by the end of Q3. Failure to do so
will result in temporary suspension of VPN access.
Section 9: Regarding the cloud restructure initiative mentioned
earlier, the final deadline for its completion is December 31st,
2026. The budget approved is $500,000.
"""

In [12]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=120
)

docs = splitter.create_documents([tricky_document])

In [13]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [15]:
prompt = ChatPromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question:
{question}
""")

In [16]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [19]:
def run_rag(chunk_size, chunk_overlap):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    docs = splitter.create_documents([tricky_document])

    vectorstore = FAISS.from_documents(docs, embeddings)

    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    rag_pipeline = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    answer = rag_pipeline.invoke(
        "What is the deadline and budget for Project Phoenix?"
    )

    return answer

In [24]:
bad_answer = run_rag(50, 10)

print("Bad Configuration Output:\n")
print(bad_answer)

Bad Configuration Output:

Unfortunately, the context provided does not include the deadline for Project Phoenix. However, it does mention that the budget approved for the project is $500,000.


In [21]:
good_answer = run_rag(350, 120)

print("Improved Configuration Output:\n")
print(good_answer)

Improved Configuration Output:

According to the context, the deadline for Project Phoenix is December 31st, 2026, and the budget approved for this project is $500,000.


## Summary
Chunking Strategy Explanation

Initially I experimented with a small chunk_size (~50) and chunk_overlap = 10.
This caused context fragmentation. Section 1 introduces the name "Project Phoenix"
while Section 9 references it indirectly as "the cloud restructure initiative
mentioned earlier".

When overlap was ten, the retriever often returned only Section 9. The model
could see the deadline and budget but could not confidently link it back to
Project Phoenix, which caused weak or incorrect answers.

To fix this, I increased the chunk_size to 350 and introduced a chunk_overlap
of 120 characters. The larger chunk size allows related information to stay
within nearby chunks, while overlap ensures that contextual references near
boundaries appear in multiple chunks.

This allows the retriever to bring both the project reference and the financial
details into the context window, enabling the model to correctly answer:

"December 31st, 2026, with a $500,000 budget."
